## Overview

Mistral Document AI offers enterprise-level document processing with OCR technology and structured data extraction. This notebook uses:

- **Mistral Document AI API** for OCR and image extraction
- **Bounding Box Annotations** for structured signature detection
- **JSON Schema** to define the expected output format

The bounding box annotation feature allows you to extract information about signatures in a structured JSON format with a single API call.

## Setup and Dependencies

In [ ]:
import base64
import json
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv
import os

## Environment Configuration

Set up your Azure Mistral Document AI credentials:

In [ ]:
# Load environment variables
load_dotenv()

# Azure Mistral Document AI endpoint and API key
AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT", "")
AZURE_MISTRAL_DOCUMENT_AI_KEY = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_KEY", "")

REQUEST_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AZURE_MISTRAL_DOCUMENT_AI_KEY}",
}

# Verify configuration
if not AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT or not AZURE_MISTRAL_DOCUMENT_AI_KEY:
    print("⚠️  Warning: Please set AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT and AZURE_MISTRAL_DOCUMENT_AI_KEY")
else:
    print("✓ Configuration loaded successfully")

## Helper Functions

In [ ]:
def encode_image_to_base64(image_path: str) -> str | None:
    """Encode an image or document file to base64 string.
    
    Args:
        image_path: Path to the image or document file
        
    Returns:
        Base64 encoded string of the file, or None if file not found
    """
    try:
        with open(image_path, "rb") as file:
            return base64.b64encode(file.read()).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None

## Signature Detection Schema

Define the JSON schema for signature detection. This schema instructs Mistral Document AI to:
- Identify handwritten signatures in the document
- Extract bounding box coordinates
- Determine confidence levels
- Assess signature characteristics

In [ ]:
SIGNATURE_BBOX_ANNOTATION_SCHEMA: dict[str, Any] = {
    "type": "json_schema",
    "json_schema": {
        "name": "signature_detection",
        "description": "Detect and analyze handwritten signatures in documents and images. A signature is a person's name written in a distinctive, personalized handwriting style, typically used for authentication or authorization purposes.",
        "schema": {
            "properties": {
                "contains_signature": {
                    "type": "boolean",
                    "description": "Whether this bounding box contains a handwritten signature (not printed text or random marks)"
                },
                "signature_type": {
                    "type": "string",
                    "description": "The type of signature detected: 'cursive' (flowing connected letters), 'printed' (hand-printed letters), 'stylized' (with flourishes/underlines), 'initials' (abbreviated signature), or 'none' (no signature present)",
                    "enum": ["cursive", "printed", "stylized", "initials", "none"]
                },
                "confidence_level": {
                    "type": "string",
                    "description": "Confidence that this is a genuine handwritten signature: 'high' (clear signature characteristics), 'medium' (some signature features), 'low' (uncertain if it's a signature)",
                    "enum": ["high", "medium", "low"]
                },
                "characteristics": {
                    "type": "string",
                    "description": "Brief description of the signature's visual appearance (e.g., 'flowing cursive in blue ink', 'printed signature in black', 'stylized with underline flourish')"
                },
                "legible": {
                    "type": "boolean",
                    "description": "Whether the signature is legible enough to potentially read the name"
                },
                "estimated_name": {
                    "type": "string",
                    "description": "The name if legible, otherwise empty string. Only provide if you can read the name with reasonable confidence."
                },
                "location_context": {
                    "type": "string",
                    "description": "Contextual information about where the signature appears (e.g., 'on signature line', 'near date field', 'bottom of page', 'next to witness label')"
                }
            },
            "required": ["contains_signature", "signature_type", "confidence_level"]
        }
    }
}

## Signature Extraction Function

Main function to extract signatures from documents using Mistral Document AI:

In [ ]:
def extract_signatures_mistral(image_path: str, include_images: bool = True) -> dict[str, Any]:
    """
    Extract handwritten signatures from an image or document using Mistral Document AI.
    
    Args:
        image_path: Path to the image or document file (supports PDF, images, etc.)
        include_images: Whether to include base64-encoded images in the response
        
    Returns:
        Dictionary containing signature extraction results with bounding box annotations
    """
    # Encode document to base64
    encoded_document = encode_image_to_base64(image_path)
    
    if not encoded_document:
        return {"error": "Failed to encode document"}
    
    # Determine file type
    file_ext = Path(image_path).suffix.lower()
    mime_types = {
        ".pdf": "application/pdf",
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".tiff": "image/tiff",
        ".tif": "image/tiff",
        ".bmp": "image/bmp"
    }
    mime_type = mime_types.get(file_ext, "application/pdf")
    
    # Construct API request payload
    payload: dict[str, Any] = {
        "model": "mistral-document-ai-2505",
        "document": {
            "type": "document_url",
            "document_url": f"data:{mime_type};base64,{encoded_document}",
        },
        "include_image_base64": str(include_images).lower(),
        "bbox_annotation_format": SIGNATURE_BBOX_ANNOTATION_SCHEMA
    }
    
    # Make API call
    try:
        response = requests.post(
            url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
            json=payload,
            headers=REQUEST_HEADERS,
            timeout=60
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        return {"error": f"API request failed: {str(e)}"}

## Process and Display Results

In [ ]:
def process_signature_results(result: dict[str, Any]) -> dict[str, Any]:
    """
    Process the raw API response and extract signature information.
    
    Args:
        result: Raw response from Mistral Document AI
        
    Returns:
        Processed signature detection results
    """
    if "error" in result:
        return result
    
    signatures_found: list[dict[str, Any]] = []
    
    for page_idx, page in enumerate(result.get("pages", [])):
        for image_idx, image in enumerate(page.get("images", [])):
            # Parse the image annotation (it's a JSON string)
            if "image_annotation" in image:
                try:
                    annotation = json.loads(image["image_annotation"])
                    props = annotation.get("properties", {})
                    
                    # Only include if it contains a signature
                    if props.get("contains_signature", False):
                        signature_info: dict[str, Any] = {
                            "page": page_idx,
                            "image_index": image_idx,
                            "bounding_box": image.get("bbox", {}),
                            "signature_type": props.get("signature_type", "unknown"),
                            "confidence_level": props.get("confidence_level", "unknown"),
                            "characteristics": props.get("characteristics", ""),
                            "legible": props.get("legible", False),
                            "estimated_name": props.get("estimated_name", ""),
                            "location_context": props.get("location_context", "")
                        }
                        
                        # Include base64 image if available
                        if "image_base64" in image:
                            signature_info["image_base64"] = image["image_base64"]
                        
                        signatures_found.append(signature_info)
                except json.JSONDecodeError:
                    continue
    
    return {
        "signatures_found": len(signatures_found),
        "signatures": signatures_found,
        "raw_response": result
    }


def print_signature_results(processed_result: dict[str, Any]) -> None:
    """
    Pretty print the signature detection results.
    
    Args:
        processed_result: Processed signature detection results
    """
    if "error" in processed_result:
        print(f"❌ Error: {processed_result['error']}")
        return
    
    print("=" * 70)
    print("SIGNATURE DETECTION RESULTS (Mistral Document AI)")
    print("=" * 70)
    print(f"\n✓ Signatures found: {processed_result['signatures_found']}")
    
    if processed_result['signatures_found'] == 0:
        print("\n⚠️  No signatures detected in the document.")
        return
    
    for idx, sig in enumerate(processed_result['signatures'], 1):
        print(f"\n{'-' * 70}")
        print(f"Signature #{idx}:")
        print(f"  📄 Page: {sig['page']}")
        print(f"  🔖 Type: {sig['signature_type']}")
        print(f"  📊 Confidence: {sig['confidence_level']}")
        print(f"  ✍️  Characteristics: {sig['characteristics']}")
        print(f"  👁️  Legible: {'Yes' if sig['legible'] else 'No'}")
        
        if sig['estimated_name']:
            print(f"  👤 Estimated name: {sig['estimated_name']}")
        
        if sig['location_context']:
            print(f"  📍 Location: {sig['location_context']}")
        
        if sig['bounding_box']:
            bbox = sig['bounding_box']
            print(f"  📐 Bounding box: x={bbox.get('x', 0)}, y={bbox.get('y', 0)}, "
                  f"width={bbox.get('width', 0)}, height={bbox.get('height', 0)}")

## Example Usage

Process a single document or image to extract signatures:

In [ ]:
# Example: Process a single image
image_path = "path/to/your/document.pdf"  # Update with your file path

# Check if file exists
if Path(image_path).exists():
    print(f"🔍 Analyzing: {image_path}\n")
    
    # Extract signatures
    raw_result = extract_signatures_mistral(image_path)
    
    # Process and display results
    processed_result = process_signature_results(raw_result)
    print_signature_results(processed_result)
    
else:
    print(f"❌ File not found: {image_path}")
    print("\nℹ️  Please update the image_path variable with a valid path to test signature extraction.")

## Batch Processing

Process multiple documents in a directory:

In [ ]:
def process_directory_mistral(directory_path: str, output_file: str = "signature_results_mistral.json") -> dict[str, Any]:
    """
    Process all images and documents in a directory and save results to JSON.
    
    Args:
        directory_path: Path to directory containing files
        output_file: Path to output JSON file
        
    Returns:
        Dictionary of results for all processed files
    """
    results: dict[str, Any] = {}
    supported_extensions = {".pdf", ".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
    
    directory = Path(directory_path)
    if not directory.exists():
        print(f"❌ Directory not found: {directory_path}")
        return results
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"📁 Found {len(files)} files to process\n")
    
    for idx, file_path in enumerate(files, 1):
        print(f"🔍 Processing {idx}/{len(files)}: {file_path.name}")
        
        try:
            raw_result = extract_signatures_mistral(str(file_path), include_images=False)
            processed_result = process_signature_results(raw_result)
            
            results[file_path.name] = processed_result
            
            if "error" in processed_result:
                print(f"  ❌ Error: {processed_result['error']}")
            else:
                print(f"  ✓ Found {processed_result['signatures_found']} signature(s)")
                
        except Exception as e:
            print(f"  ❌ Exception: {str(e)}")
            results[file_path.name] = {"error": str(e)}
    
    # Save results to JSON (without base64 images to keep file size reasonable)
    try:
        # Remove raw_response to reduce file size
        clean_results: dict[str, Any] = {}
        for filename, result in results.items():
            if "raw_response" in result:
                result_copy = result.copy()
                result_copy.pop("raw_response", None)
                clean_results[filename] = result_copy
            else:
                clean_results[filename] = result
        
        with open(output_file, "w") as f:
            json.dump(clean_results, f, indent=2)
        
        print(f"\n✓ Results saved to: {output_file}")
    except Exception as e:
        print(f"\n❌ Failed to save results: {str(e)}")
    
    return results


# Example usage (uncomment to run)
# results = process_directory_mistral("./testdata", "signature_results_mistral.json")

## Visualization (Optional)

Draw bounding boxes on images to visualize detected signatures:

In [ ]:
# Optional: Install PIL/Pillow for visualization
# !pip install pillow

try:
    from PIL import Image, ImageDraw
    
    def visualize_signatures_mistral(image_path: str, result: dict[str, Any], output_path: str | None = None) -> Image.Image | None:
        """
        Draw bounding boxes on image for detected signatures.
        
        Args:
            image_path: Path to original image
            result: Processed signature detection result
            output_path: Path to save annotated image (optional)
            
        Returns:
            PIL Image object or None if error
        """
        if "error" in result:
            print(f"Cannot visualize - error in result: {result['error']}")
            return None
        
        # For PDFs, we can't directly visualize - would need to convert to images first
        if image_path.lower().endswith('.pdf'):
            print("⚠️  PDF visualization requires conversion to images first.")
            print("Consider using a library like pdf2image for PDF visualization.")
            return None
        
        # Open image
        img = Image.open(image_path)
        draw = ImageDraw.Draw(img)
        
        # Draw bounding boxes for each signature
        for sig in result.get('signatures', []):
            bbox = sig.get('bounding_box', {})
            
            if not bbox:
                continue
            
            # Extract bbox coordinates (assuming they're in pixels)
            x = bbox.get('x', 0)
            y = bbox.get('y', 0)
            w = bbox.get('width', 0)
            h = bbox.get('height', 0)
            
            # Choose color based on confidence
            confidence = sig.get('confidence_level', 'low')
            color_map = {
                'high': 'green',
                'medium': 'orange',
                'low': 'red'
            }
            color = color_map.get(confidence, 'blue')
            
            # Draw rectangle
            draw.rectangle([x, y, x+w, y+h], outline=color, width=3)
            
            # Add label
            label = f"{sig.get('signature_type', 'signature')} ({confidence})"
            draw.text((x, y-20), label, fill=color)
        
        # Display or save
        if output_path:
            img.save(output_path)
            print(f"✓ Annotated image saved to: {output_path}")
        
        return img
    
    print("✓ Visualization functions loaded")
    
except ImportError:
    print("⚠️  PIL/Pillow not installed. Run: pip install pillow")
    print("Visualization functions will not be available.")

## Notes and Best Practices

**Requirements:**
- Azure Mistral Document AI endpoint with API key
- Environment variables configured in `.env` file:
  - `AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT`
  - `AZURE_MISTRAL_DOCUMENT_AI_KEY`

**Supported File Formats:**
- PDF documents (`.pdf`)
- Images (`.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`, `.tif`)

**Tips:**
- Use high-resolution images for best signature detection results
- The bounding box annotation schema ensures structured output
- Set `include_image_base64=False` when processing many files to reduce response size
- Document annotations are currently limited to 8 pages
- The API extracts text and identifies bounding boxes automatically

**Performance Considerations:**
- API calls may take longer for large documents or high-resolution images
- Consider batching with delays between requests for large volumes
- Cache results when processing the same document multiple times

**Key Differences from OpenAI Approach:**
- Mistral uses bounding box annotations for structured extraction
- Single API call extracts both OCR and signature annotations
- No need for separate vision model calls
- Built-in support for document structure (pages, images, text)
- Predefined schema ensures consistent output format

## Comparison with OpenAI Vision Approach

| Feature | Mistral Document AI | OpenAI Vision |
|---------|---------------------|---------------|
| **API Calls** | Single call with bbox annotations | Chat completion with vision |
| **Structure** | Predefined JSON schema | Prompt-based extraction |
| **Bounding Boxes** | Native support | Percentage-based estimation |
| **Document Processing** | Built-in OCR + annotations | Requires separate processing |
| **Multi-page PDFs** | Native support (up to 8 pages) | Requires page-by-page conversion |
| **Output Format** | Strictly typed via schema | Flexible but variable |
| **Image Extraction** | Automatic with base64 encoding | Manual handling required |

**Use Mistral Document AI when:**
- Processing multi-page documents
- Need structured, predictable output
- Want automatic OCR + annotation in one call
- Working with complex document layouts

**Use OpenAI Vision when:**
- Need flexible, conversational analysis
- Working primarily with single images
- Require detailed natural language descriptions
- Want more interpretive analysis